# 01 — Bronze Ingestion

Reads the raw `vgsales.csv` from ADLS and writes it as a **Delta table** in the Bronze layer.

Bronze = raw data with no business logic, just ingestion metadata added.

In [ ]:
# Read raw CSV as-is
df_raw = spark.read.csv(
    "/mnt/gaming/raw/vgsales.csv",
    header=True,
    inferSchema=True
)
print(f"Raw row count : {df_raw.count()}")
print(f"Columns       : {df_raw.columns}")
df_raw.printSchema()
display(df_raw.limit(5))

In [ ]:
from pyspark.sql.functions import current_timestamp, lit, input_file_name

df_bronze = (
    df_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
    .withColumn("_layer",       lit("bronze"))
)
display(df_bronze.limit(3))

In [ ]:
# Write to Bronze Delta table
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gaming_bronze.vg_sales_raw")

print("Bronze Delta table written ✅")
spark.sql("SELECT COUNT(*) AS total_rows FROM gaming_bronze.vg_sales_raw").show()

In [ ]:
spark.sql('OPTIMIZE gaming_bronze.vg_sales_raw')
print("OPTIMIZE complete ✅")

In [ ]:
display(spark.sql(
    'SELECT Platform, Genre, COUNT(*) AS titles '
    'FROM gaming_bronze.vg_sales_raw '
    'GROUP BY Platform, Genre ORDER BY titles DESC LIMIT 20'
))